# Synthetic Training Notebook

This notebook trains **sample** models on **synthetic data** to demonstrate the pipeline used in the hackathon.
The metrics produced here are **not** the original hackathon results and should not be used to claim the ~83% accuracy.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [ ]:
maintenance_df = pd.read_csv('../data/training/maintenance_training.csv')
demand_df = pd.read_csv('../data/training/demand_training.csv')
maintenance_df.head()


## Predictive Maintenance (Random Forest)

Target: `needs_service` (binary).


In [ ]:
features = [
    'mileage',
    'load_cycles',
    'service_history_days',
    'avg_engine_temp_c',
    'avg_fuel_consumption_l_100km',
]
X = maintenance_df[features]
y = maintenance_df['needs_service']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f'Sample accuracy: {acc:.2%}')
print(classification_report(y_test, preds))


## Demand Forecasting (Logistic Regression)

Target: `shortage_risk` (binary).


In [ ]:
categorical = ['neighborhood', 'day_of_week', 'weather']
numeric = ['is_holiday', 'demand_cylinders', 'capacity_cylinders', 'reported_deliveries']

X = demand_df[categorical + numeric]
y = demand_df['shortage_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore'), categorical)],
    remainder='passthrough',
)

clf = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=1000)),
    ]
)

clf.fit(X_train, y_train)
preds = clf.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f'Sample accuracy: {acc:.2%}')
print(classification_report(y_test, preds))
